# (1) import libraries

In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from scipy.stats import multivariate_normal
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import time
from torchvision import datasets, transforms
torch.manual_seed(42)
np.random.seed(42)

# (2) data preparation

In [2]:
# Load MNIST datatransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
for images, labels in train_loader:
    print('X_train shape:', images.shape)
    print(train_dataset.train_labels.shape[0], 'train samples')
    break

# (3) Model definition

In [3]:
# Define MLP model using PyTorchclass MLP(nn.Module):
    def __init__(self, input_size=784, hidden_sizes=[512, 256], num_classes=10, dropout=0.5):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(dropout)
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.fc = nn.Sequential(*layers)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc(x)
        return x

# Initialize modeldevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# (4) Training

In [4]:
# Training loopnum_epochs = 30
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (i + 1) % 100 == 0:
            print[f'Epoch [{epoch+1}/30], Step [{i+1}/{(len(train_dataset)//batch_size)}], Loss: {running_loss / 100:.4f}']
            running_loss = 0.0
    print(f'Epoch [{epoch+1}/30] completed')
start_time = time.time()

# (5) 3D Visualization

In [5]:
with torch.no_grad():
    data_iter = iter(train_loader)
    images, labels = next(data_iter)
    images = images[:20].to(device)
    labels = labels[:20]
model.eval()
outputs = model(images)
_, predicted = torch.max(outputs, 1)
plt.figure(figsize=(12, 4))
for i in range(20):
    plt.subplot(2, 10, i+1)
    img = images[i].cpu().squeeze()
    plt.imshow(img.numpy(), cmap='gray')
    plt.title(f'Pred: {predicted[i].item()}, True: {labels[i].item()}')
    plt.axis('off')
plt.tight_layout()
plt.show()

# (6) Miscellaneous- Training completed!- Model saved for future use- 3D visualization of predictions